In [1]:
import numpy as np
import pandas as pd
import random


(a) Represent HMM Parameters # This section defines the three core probability matrices of the HMM.



In [16]:
import numpy as np
import pandas as pd
import random


# 1. HMM Parameter Definitions


# Define the set of hidden states (phonemes) and observations
PHONEMES = ['/s/', '/p/', '/ie:/', '/tS/']
OBSERVATIONS = ['Energy', 'Pitch', 'Duration']

# Initial Probabilities (Pi)
# Given: P(start at /s/) = 1.0
initial_probs = {
    '/s/': 1.0,
    '/p/': 0.0,
    '/ie:/': 0.0,
    '/tS/': 0.0
}

# Transition Probabilities (A) - P(State_t+1 | State_t)
# Extracted from the provided table.
transition_probs = {
    '/s/':   {'/s/': 0.1, '/p/': 0.8, '/ie:/': 0.1, '/tS/': 0.0},
    '/p/':   {'/s/': 0.0, '/p/': 0.1, '/ie:/': 0.8, '/tS/': 0.1},
    '/ie:/': {'/s/': 0.0, '/p/': 0.0, '/ie:/': 0.2, '/tS/': 0.8},
    '/tS/':  {'/s/': 0.2, '/p/': 0.0, '/ie:/': 0.0, '/tS/': 0.8}
}

# Emission Probabilities (B) - P(Observation | State)
# Extracted from the provided table.
emission_probs = {
    '/s/':   {'Energy': 0.7, 'Pitch': 0.2, 'Duration': 0.1},
    '/p/':   {'Energy': 0.5, 'Pitch': 0.3, 'Duration': 0.2},
    '/ie:/': {'Energy': 0.3, 'Pitch': 0.5, 'Duration': 0.2},
    '/tS/':  {'Energy': 0.4, 'Pitch': 0.4, 'Duration': 0.2}
}


# 2 Function to neatly display the matrices

def display_hmm_parameters():
    """Neatly displays the HMM parameters (Pi, A, B) using pandas tables."""
    print("--- (b) Display of HMM Parameters ---")

    # 1. Display Initial Probabilities
    print("\nInitial Probabilities (Pi):")
    pi_df = pd.Series(initial_probs, name='P(start)')
    print(pi_df.to_markdown())

    # 2. Display Transition Matrix (A)
    print("\nTransition Probabilities (A):")
    A_df = pd.DataFrame(transition_probs).T
    A_df.index.name = 'From'
    A_df.columns.name = 'To'
    print(A_df.to_markdown())

    # 3. Display Emission Matrix (B)
    print("\nEmission Probabilities (B):")
    B_df = pd.DataFrame(emission_probs).T
    B_df.index.name = 'Phoneme'
    B_df.columns.name = 'Observation'
    print(B_df.to_markdown())



# (c) Program to generate a single sequence

def generate_sequence(num_steps=4):
    """
    Generates a sequence of phonemes (hidden states) and corresponding
    acoustic observations based on the HMM probabilities.
    """

    phoneme_sequence = []
    observation_sequence = []

    # --- Determine the starting state (Initialization) ---
    states = list(initial_probs.keys())
    probabilities = list(initial_probs.values())

    # Use numpy.random.choice to select the starting state based on initial_probs
    current_state = np.random.choice(states, p=probabilities)

    # ---  Loop for the required number of steps (4 for 'speech') ---
    for step in range(num_steps):

        # A. Emission Step: Generate Observation P(Obs | State)
        obs_labels = list(emission_probs[current_state].keys())
        obs_dist = list(emission_probs[current_state].values())

        # Randomly choose an observation based on the emission probabilities
        observation = np.random.choice(obs_labels, p=obs_dist)

        # Record the current state and the emitted observation
        phoneme_sequence.append(current_state)
        observation_sequence.append(observation)

        # B. Transition Step: Move to Next State P(State_t+1 | State_t)
        if step == num_steps - 1:
            break # Stop after the last phoneme is emitted

        trans_labels = list(transition_probs[current_state].keys())
        trans_dist = list(transition_probs[current_state].values())

        # Select the next state based on the transition probabilities
        next_state = np.random.choice(trans_labels, p=trans_dist)

        # Update the current state for the next step
        current_state = next_state

    return phoneme_sequence, observation_sequence

#  Main Execution

if __name__ == '__main__':
    # Execute Task (b)
    display_hmm_parameters()

    # Execute Task (c)
    print("\n" + "="*50)
    print("--- (c) HMM Sequence Generation (4 steps) ---")

    generated_phonemes, generated_observations = generate_sequence(num_steps=4)

    print(f"\nGenerated Phoneme Sequence: {generated_phonemes}")
    print(f"Corresponding Acoustic Observations: {generated_observations}")
    print("="*50)

--- (b) Display of HMM Parameters ---

Initial Probabilities (Pi):
|       |   P(start) |
|:------|-----------:|
| /s/   |          1 |
| /p/   |          0 |
| /ie:/ |          0 |
| /tS/  |          0 |

Transition Probabilities (A):
| From   |   /s/ |   /p/ |   /ie:/ |   /tS/ |
|:-------|------:|------:|--------:|-------:|
| /s/    |   0.1 |   0.8 |     0.1 |    0   |
| /p/    |   0   |   0.1 |     0.8 |    0.1 |
| /ie:/  |   0   |   0   |     0.2 |    0.8 |
| /tS/   |   0.2 |   0   |     0   |    0.8 |

Emission Probabilities (B):
| Phoneme   |   Energy |   Pitch |   Duration |
|:----------|---------:|--------:|-----------:|
| /s/       |      0.7 |     0.2 |        0.1 |
| /p/       |      0.5 |     0.3 |        0.2 |
| /ie:/     |      0.3 |     0.5 |        0.2 |
| /tS/      |      0.4 |     0.4 |        0.2 |

--- (c) HMM Sequence Generation (4 steps) ---

Generated Phoneme Sequence: [np.str_('/s/'), np.str_('/p/'), np.str_('/ie:/'), np.str_('/tS/')]
Corresponding Acoustic Obse

HMM Inference

This implementation performs Sequence Generation by probabilistically creating a path of hidden phonemes and their associated acoustic observations.

Model Purpose: The HMM is designed to model the temporal sequence of phonemes in the word 'speech' by simulating how one phoneme leads to the next (Transition) and what measurable acoustic properties (like Energy or Pitch) are produced by that phoneme (Emission).

Key Assumptions (Markov Property): The model relies on two independence rules:

Transition: The next phoneme is chosen based only on the current phoneme ($P(\text{State}_{t+1} \mid \text{State}_t)$).

Emission: The acoustic observation is chosen based only on the current phoneme ($P(\text{Obs}_t \mid \text{State}_t)$).

Probabilistic Meaning:

Transition Dominance: The transition matrix heavily favors the linguistically correct sequence ($\text{/s/} \to \text{/p/} \to \text{/ie:/} \to \text{/tS/}$), ensuring the generated sequence is a plausible pronunciation of the word.

Acoustic Link: The emission probabilities are realistic: the vowel $\text{/ie:/}$ has the highest chance of emitting Pitch (as it's voiced), while the fricative $\text{/s/}$ has the highest chance of emitting Energy (as it is unvoiced friction).

Real-World Context: The parameters defined here ($\Pi, A, B$) are the foundation used in actual speech recognition systems (like Viterbi decoding) to find the most likely phoneme sequence given a recorded set of acoustic observations.